# 19_final_training_data — 최종 학습데이터 만들기

**한 줄 요약:** 한 분자당 [SMILES] + [fingerprint 1024] + [WEKA 선택 descriptor] + [정답 potency] 로 된 **최종 표**를 만든다.
**용어:** fingerprint=분자 구조를 0/1 비트 1024개로 나타낸 지문 / ECFP4=대표적 지문 종류.
**큰 흐름:** ① 준비 → ② 재료 읽기 → ③ 지문 계산 → ④ 네 조각 결합·저장

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다. 앞 셀에서 만든 값을 뒤 셀이 쓰므로 **순서대로**.
> - 각 코드 셀은 **[① 무슨 작업인지 설명] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서로 놓았다.
>   ③은 그 셀에 **처음 나온** 함수·문법을 잘게 푼 것(이미 나온 건 반복 안 함).
> - 코드 줄 뒤 `# ...` 은 **주석**(설명)이라 실행에 영향 없음.

### 셀 1 — 준비 + fingerprint '생성기' 만들기
라이브러리를 가져오고, 지문 생성기를 미리 하나 만들어 재사용한다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

FP_BITS = 1024
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)

🔎 **코드 뜯어보기 (셀 1)** *(import/as/from은 18과 동일)*
- `rdFingerprintGenerator` : RDKit에서 fingerprint(지문)를 만드는 도구 모음.
- `FP_BITS = 1024` : 지문 길이(비트 수)를 1024로 정해 변수에 저장. 뒤에서 이 값을 재사용.
- `GetMorganGenerator(radius=2, fpSize=FP_BITS)` : **ECFP4 지문 생성기**를 만든다. `radius=2`(원자 주변을 2단계까지 봄), `fpSize`(비트 수). **`이름=값`** 처럼 넘기는 건 **키워드 인자**(어떤 설정인지 이름으로 지정).

### 셀 2 — 재료 두 개 읽기
전체 descriptor Excel과, WEKA가 고른 descriptor '이름 목록'을 읽는다.

In [ ]:
# 입력: (1) 전체 descriptor Excel  (2) WEKA로 고른 descriptor 목록(헤더만)
FULL = 'data/HSD17B13_1to1_descriptors.xlsx'
FILT = 'data/HSD17B13_1to1_descriptors_weka_filtered.csv'

full = pd.read_excel(FULL)
sel_cols = [c for c in pd.read_csv(FILT, nrows=0).columns if c.lower() != 'potency']
print('WEKA 선택 descriptor:', len(sel_cols), '개')

miss = [c for c in sel_cols if c not in full.columns]
assert not miss, ('full에 없는 descriptor: %s' % miss)
print('전체 화합물:', len(full), '| potency 분포:', dict(full.potency.value_counts()))

🔎 **코드 뜯어보기 (셀 2)**
- `pd.read_excel(...)` : 엑셀 파일을 표로 읽기(CSV용 read_csv의 엑셀판).
- `pd.read_csv(FILT, nrows=0)` : **nrows=0** = 데이터 없이 **열 이름(헤더)만** 읽기(값은 필요 없고 이름만 필요).
- `[c for c in ... .columns if c.lower() != 'potency']` : 리스트 컴프리헨션에 **if 조건** 추가 — '이름 소문자가 potency가 아닌' 열만 골라 리스트로. `.lower()`=소문자로.
- `assert 조건, 메시지` : **assert**는 '이 조건이 참이 아니면 여기서 멈추고 메시지 표시'. 오타·불일치를 **미리 잡는 안전장치**. `not miss` = 'miss 리스트가 비었으면 참'.

### 셀 3 — 각 분자를 fingerprint(1024비트)로 변환
분자를 하나씩 지문으로 바꿔 모으고, 표로 만든다.

In [ ]:
# canonical SMILES -> ECFP4 fingerprint(1024 bit)
fp_rows, keep = [], []
for i, smi in enumerate(full['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        continue
    fp_rows.append(gen_ecfp.GetFingerprintAsNumPy(m))
    keep.append(i)
FP = pd.DataFrame(np.vstack(fp_rows).astype(np.int8),
                  columns=['fp_%04d' % j for j in range(FP_BITS)])
base = full.iloc[keep].reset_index(drop=True)
print('fingerprint 계산 완료:', FP.shape, '| 유효 화합물', len(base))

🔎 **코드 뜯어보기 (셀 3)** *(for/enumerate/MolFromSmiles/append/iloc는 18과 동일)*
- `gen_ecfp.GetFingerprintAsNumPy(m)` : 셀1에서 만든 생성기로 분자 m의 지문을 **numpy 배열(0/1 1024개)** 로 계산.
- `np.vstack(fp_rows)` : 여러 배열을 **위아래로 쌓아** 하나의 큰 표(2차원 배열)로. (v=vertical=수직)
- `.astype(np.int8)` : 자료형을 **int8(작은 정수)** 로 바꿔 용량 절약(0/1만 담으면 충분).
- `['fp_%04d' % j for j in range(FP_BITS)]` : **range(1024)** = 0~1023 숫자열. 각 j로 `fp_0000`~`fp_1023` 열 이름을 만든다(`%04d`=4자리 0채움).

### 셀 4 — 네 조각을 결합해 최종 표 만들기 → CSV·Excel 저장
SMILES·지문·descriptor·정답을 좌우로 붙이고 두 형식으로 저장한다.

In [ ]:
# 최종 조립: canonical_smiles + fingerprint(1024) + descriptor(선택) + potency
final = pd.concat([
    base[['canonical_smiles']].reset_index(drop=True),
    FP.reset_index(drop=True),
    base[sel_cols].reset_index(drop=True),
    base[['potency']].reset_index(drop=True),
], axis=1)
print('최종 학습데이터 shape:', final.shape,
      '(= canonical_smiles 1 + fp %d + desc %d + potency 1)' % (FP_BITS, len(sel_cols)))
print('구성 확인 → 첫 열:', final.columns[0], '| 마지막 열:', final.columns[-1])

OUT_CSV = 'data/HSD17B13_final_training_1to1.csv'
OUT_XLSX = 'data/HSD17B13_final_training_1to1.xlsx'
final.to_csv(OUT_CSV, index=False)
final.to_excel(OUT_XLSX, index=False)
print('저장 완료:')
print('  CSV :', OUT_CSV)
print('  XLSX:', OUT_XLSX)

🔎 **코드 뜯어보기 (셀 4)** *(concat/to_csv/to_excel은 18과 동일)*
- `pd.concat([A, B, C, D], axis=1)` : 네 개 표를 **좌우로** 순서대로 붙임 → 왼쪽부터 SMILES, 지문, descriptor, 정답.
- `.reset_index(drop=True)` : 각 조각의 **행 번호를 0,1,2…로 새로 매김**. 안 그러면 번호가 안 맞아 붙일 때 어긋날 수 있음(drop=True=옛 번호 버림).